# Module 11 · 5：影片下游 — VideoMAE 動作辨識微調

> 影片資料（M09 影片小節）整理好後，最常見的下游是**動作辨識**：
> 把預訓練 VideoMAE 微調到你自己的動作類別。

## 學習目標
- 複習影片推論流程（抽樣 → processor → 模型）。
- 認識 **VideoMAE 微調**的資料格式 `(N,T,C,H,W)` + 標籤，與 `Trainer` 設定。

In [ ]:
import numpy as np
try:
    import torch
    HAS = True
except Exception:
    HAS = False
    print("需 `uv sync --extra multimodal --extra train`。說明仍可閱讀。")

## A. 推論複習（zero-extra-training）

用在 Kinetics-400 上微調過的 VideoMAE，可直接辨識 400 種動作（見 M09 影片 03）。

In [ ]:
if HAS:
    try:
        from transformers import AutoImageProcessor, VideoMAEForVideoClassification
        proc = AutoImageProcessor.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
        model = VideoMAEForVideoClassification.from_pretrained("MCG-NJU/videomae-base-finetuned-kinetics")
        clip = [(np.random.RandomState(i).rand(224, 224, 3)*255).astype(np.uint8) for i in range(16)]
        inputs = proc(clip, return_tensors="pt")
        with torch.no_grad():
            logits = model(**inputs).logits
        print("輸入:", tuple(inputs["pixel_values"].shape), "(N,T,C,H,W)")
        print("預測:", model.config.id2label[int(logits.argmax(-1))], "（合成影格僅示意）")
    except Exception as e:
        print("（未能下載 VideoMAE，略過）", e)

## B. 微調到自己的動作類別

資料格式：每筆 = 一段 clip 的 16 張抽樣影格 `(T,C,H,W)` + 動作標籤。
換成 `VideoMAEForVideoClassification(num_labels=你的類別數)` 後，用同一套 `Trainer`。

In [ ]:
if HAS:
    print("""微調設定骨架（概念）：

    from transformers import (VideoMAEImageProcessor,
        VideoMAEForVideoClassification, TrainingArguments, Trainer)

    model = VideoMAEForVideoClassification.from_pretrained(
        "MCG-NJU/videomae-base",            # 用未微調的 base 當起點
        num_labels=NUM_ACTIONS,
        ignore_mismatched_sizes=True)

    # dataset: 每筆 = 16 張抽樣影格(經 processor) + 標籤
    args = TrainingArguments(output_dir="...", num_train_epochs=5,
                             per_device_train_batch_size=2, learning_rate=5e-5)
    Trainer(model=model, args=args, train_dataset=ds).train()
    """)
    print("→ 影片微調算力較重（5 維張量），實務多用 GPU + 少量影格 + LoRA 等技巧。")

## 小結

| 階段 | 資料 | 模型 |
|:--|:--|:--|
| 推論 | `(N,T,C,H,W)` | VideoMAE (Kinetics 微調版) |
| 微調 | `(N,T,C,H,W)` + 動作標籤 | VideoMAE base + `Trainer` |

影片是四模態中算力最重的；資料前處理（抽樣！）對效率影響最大。